# Google Reviews Extraction
Flatten all `hotel_{hotel_id}_reviews.json` from `goorawling/outputs/` and merge with `hotel_filtered.csv`.

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path('../goorawling/outputs')
HOTEL_FILTERED_CSV = Path('../data/hotel_filtered.csv')
OUTPUT_CSV = Path('../data/reviews_merged.csv')

In [ ]:
# Collect all hotel_{id}_reviews.json files (skip all_hotels_reviews.json)
json_files = sorted(OUTPUTS_DIR.glob('hotel_*_reviews.json'))
print(f'Found {len(json_files)} hotel review files')

In [ ]:
def parse_hotel_json(path: Path) -> list[dict]:
    """Parse a single hotel review JSON into a flat list of review dicts."""
    hotel_id = int(re.search(r'hotel_(\d+)_reviews', path.name).group(1))

    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    meta = data.get('metadata', {})
    reviews_by_place = data.get('reviews_by_place', {})

    rows = []
    for place_id, reviews in reviews_by_place.items():
        for review in reviews:
            ef = review.get('edge_fields', {})
            rv_meta = ef.get('metadata', {})
            rv_section = ef.get('review_section', {})
            rv_text = rv_section.get('review_text', {})
            aspect = rv_section.get('aspect_rating', {})

            # Parse numeric rating (e.g. "4/5" -> 4)
            raw_rating = rv_meta.get('rating', '')
            try:
                rating_num = int(raw_rating.split('/')[0])
            except (ValueError, AttributeError):
                rating_num = None

            row = {
                'hotel_id': hotel_id,
                'scrape_timestamp': meta.get('timestamp'),
                'source_url': meta.get('source_url'),
                'place_id': place_id,
                'review_id': rv_meta.get('review_id'),
                'reviewer_name': ef.get('reviewer_name'),
                'reviewer_info': ef.get('reviewer_info'),
                'rating': rating_num,
                'rating_time': rv_meta.get('rating_time'),
                'platform': rv_meta.get('platform'),
                'review_text': rv_text.get('text'),
                'review_lang': rv_text.get('lang'),
                'image_count': len(ef.get('image_urls', [])),
                # Aspect ratings (Vietnamese keys mapped to English)
                'trip_type': aspect.get('Loại chuyến đi'),
                'traveler_group': aspect.get('Nhóm khách du lịch'),
                'room_rating': aspect.get('Phòng'),
                'service_rating': aspect.get('Dịch vụ'),
                'location_rating': aspect.get('Vị trí'),
                'food_rating': aspect.get('Đồ ăn'),
                'atmosphere_rating': aspect.get('Không khí'),
                'highlights': aspect.get('Điểm nổi bật của khách sạn'),
                'notable_mentions': aspect.get('Những điểm đáng chú ý'),
                'walkability': aspect.get('Nơi có thể đi bộ'),
                'nearby_activities': aspect.get('Hoạt động ở khu vực lân cận'),
            }
            rows.append(row)
    return rows

In [ ]:
# Parse all files
all_rows = []
errors = []

for path in json_files:
    try:
        all_rows.extend(parse_hotel_json(path))
    except Exception as e:
        errors.append((path.name, str(e)))

reviews_df = pd.DataFrame(all_rows)
print(f'Total reviews parsed: {len(reviews_df)}')
print(f'Unique hotels: {reviews_df["hotel_id"].nunique()}')
if errors:
    print(f'\nFiles with errors ({len(errors)}):')
    for name, err in errors:
        print(f'  {name}: {err}')

In [ ]:
reviews_df.head(3)

In [ ]:
# Load hotel_filtered.csv and merge
hotels_df = pd.read_csv(HOTEL_FILTERED_CSV)
print(f'hotel_filtered.csv: {len(hotels_df)} rows, columns: {list(hotels_df.columns[:8])} ...')

merged_df = reviews_df.merge(hotels_df, on='hotel_id', how='left')
print(f'\nMerged: {len(merged_df)} rows')
print(f'Reviews with hotel metadata: {merged_df["hotel_name"].notna().sum()}')
print(f'Reviews without match in hotel_filtered: {merged_df["hotel_name"].isna().sum()}')

In [ ]:
# Quick summary
summary = (
    merged_df.groupby('hotel_id')
    .agg(
        hotel_name=('hotel_name', 'first'),
        city=('city', 'first'),
        star_rating=('star_rating', 'first'),
        distance2coastline=('distance2coastline', 'first'),
        review_count=('review_id', 'count'),
        avg_rating=('rating', 'mean'),
    )
    .reset_index()
    .sort_values('review_count', ascending=False)
)
print(f'Hotel summary ({len(summary)} hotels):')
summary.head(10)

In [ ]:
# Save to CSV
merged_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved {len(merged_df)} rows to {OUTPUT_CSV}')